In [0]:
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

In [0]:
%sql
SELECT DISTINCT id AS idRegistro,
       nome_bairro AS nomeBairro,
       latitude AS latitude,
       longitude AS longitude

FROM bronze.climapetropolis.registros

In [0]:
%sql
SELECT *
FROM bronze.climapetropolis.registros

In [0]:
%sql
SELECT id AS idRegistro,
       get_json_object(dados_meteorologicos, '$.weather[0].description') as condicaoClima,
       get_json_object(dados_meteorologicos, '$.main.temp') as temperaturaAtual,
       get_json_object(dados_meteorologicos, '$.main.feels_like') as sensacaoTermica,
       get_json_object(dados_meteorologicos, '$.main.temp_max') as tempMax,
       get_json_object(dados_meteorologicos, '$.main.temp_min') as tempMin,
       get_json_object(dados_meteorologicos, '$.main.pressure') as pressaoAr,
       get_json_object(dados_meteorologicos, '$.main.temp_max') as tempMax,
       get_json_object(dados_meteorologicos, '$.main.humidity') as umidadeAr,
       get_json_object(dados_meteorologicos, '$.wind.speed') as velocidadeVento,
       get_json_object(dados_meteorologicos, '$.clouds.all') as porcentagemNuvens,
       CAST(get_json_object(dados_meteorologicos, '$.main.pressure') AS DOUBLE) AS pressaoAtmosfericaGeral,
       CAST(get_json_object(dados_meteorologicos, '$.main.sea_level') AS DOUBLE) AS pressaoAtmosfericaNivelMar,
       CAST(get_json_object(dados_meteorologicos, '$.main.grnd_level') AS DOUBLE) AS pressaoAtmosfericaNivelSolo,
       COALESCE(CAST(get_json_object(dados_meteorologicos, '$.rain.1h') AS DOUBLE), 0.0) AS chuvaMilimetrosPorHora,
       CAST(get_json_object(dados_meteorologicos, '$.visibility') AS INT) AS visibilidadeMetros,
       CAST(get_json_object(dados_meteorologicos, '$.wind.speed') AS DOUBLE) AS velocidadeVento,
       CAST(get_json_object(dados_meteorologicos, '$.wind.deg') AS INT) AS direcaoVentoGraus,
       CAST(get_json_object(dados_meteorologicos, '$.wind.gust') AS DOUBLE) AS rajadaVento,
       from_utc_timestamp(from_unixtime(CAST(get_json_object(dados_meteorologicos, '$.sys.sunrise') AS BIGINT)), 'America/Sao_Paulo') AS nascerDoSol,
       from_utc_timestamp(from_unixtime(CAST(get_json_object(dados_meteorologicos, '$.sys.sunset') AS BIGINT)), 'America/Sao_Paulo') as porDoSol,
       from_utc_timestamp(from_unixtime(CAST(get_json_object(dados_meteorologicos, '$.dt') AS BIGINT)), 'America/Sao_Paulo') AS data_hora_legivel,
       data_ingestao AS dtIngestao
       
FROM bronze.climapetropolis.registros

In [0]:
%sql

--salvar isso pra gold
SELECT 
    -- Arredonda para a hora cheia
    date_trunc('hour', from_utc_timestamp(
        from_unixtime(CAST(get_json_object(dados_meteorologicos, '$.dt') AS BIGINT)), 
        'America/Sao_Paulo'
    )) AS hora_cheia,
    nome_bairro,
    -- Soma o volume de chuva daquela hora
    SUM(COALESCE(CAST(get_json_object(dados_meteorologicos, '$.rain.1h') AS DOUBLE), 0.0)) AS chuva_total_hora
FROM bronze.climapetropolis.registros
GROUP BY 1, nome_bairro
ORDER BY hora_cheia DESC;

In [0]:
%sql
SELECT 
    -- 1. Converte o timestamp Unix para timestamp padrão
    -- 2. Converte de UTC para o fuso horário do Brasil
    from_utc_timestamp(
        from_unixtime(CAST(get_json_object(dados_meteorologicos, '$.dt') AS BIGINT)), 
        'America/Sao_Paulo'
    ) AS data_hora_legivel
FROM bronze.climapetropolis.registros